In [1]:
from functools import partial
from torch.utils.data import DataLoader
import torch
from omegaconf.listconfig import ListConfig
from data_loader import TemporalDataset, SingleInferenceDataset
from preprocessors.biovil_t import load_biovil_preprocessor
from preprocessors.cxr_clip import load_cxr_clip_preprocessor
from collate.biovil_t import collate_biovil_temporal, collate_biovil_single
from collate.cxr_clip import collate_cxr_clip_single

/opt/anaconda3/envs/cxr_models/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ---- Load datasets
single_ds = SingleInferenceDataset("/Volumes/TM/Single/single_clean.csv")
temporal_ds = TemporalDataset("/Volumes/TM/Temporal/temporal_clean.csv")

In [6]:
# ---- Load preprocessors
biovil_image_processor = load_biovil_preprocessor(
    "microsoft/BiomedVLP-BioViL-T",
    mode="image"
)

biovil_tokenizer = load_biovil_preprocessor(
    "microsoft/BiomedVLP-BioViL-T",
    mode="text"
)

# ---- Build loaders
single_biovil_loader = DataLoader(
    single_ds,
    batch_size=8,
    shuffle=False,
    num_workers=0,
    collate_fn=partial(
        collate_biovil_single,
        image_processor=biovil_image_processor,
        tokenizer=biovil_tokenizer,
    ),
)

temporal_biovil_loader = DataLoader(
    temporal_ds,
    batch_size=4,
    shuffle=False,
    num_workers=0,
    collate_fn=partial(
        collate_biovil_temporal,
        image_processor=biovil_image_processor,
        tokenizer=biovil_tokenizer,
    ),
)

In [8]:
ckpt_path = "/Users/taha/Desktop/UNI/ECM 3401/interpretable-cxr-models/models/cxr_clip/r50_m.tar"

ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)

image_transforms, normalize_mode = load_cxr_clip_preprocessor(ckpt)

# For CXR-CLIP single inference:
single_cxrclip_loader = DataLoader(
    single_ds,
    batch_size=16,
    shuffle=False,
    num_workers=4,
    collate_fn=partial(collate_cxr_clip_single, image_transforms=image_transforms, normalize_mode=normalize_mode),
)

In [11]:
batch = next(iter(temporal_biovil_loader))
sample = single_ds[0]
print(sample.keys())
print(sample["image_path"])
print(sample["text"][:200])

batch = next(iter(single_biovil_loader))
sample = single_ds[0]
print(sample.keys())
print(sample["image_path"])
print(sample["text"][:200])

dict_keys(['image', 'text', 'image_path', 'subject_id', 'study_id', 'dicom_id'])
/Volumes/TM/Single/images/p10/p10001217/s52067803/a917c883-720a5bbf-02c84fc6-98ad00ac-c562ff80.jpg
                                 FINAL REPORT
 HISTORY:  Multiple sclerosis, presenting with flaring fever.
 
 TECHNIQUE:  Frontal and lateral views of the chest.
 
 COMPARISON:  None.
 
 FINDINGS:
 

dict_keys(['image', 'text', 'image_path', 'subject_id', 'study_id', 'dicom_id'])
/Volumes/TM/Single/images/p10/p10001217/s52067803/a917c883-720a5bbf-02c84fc6-98ad00ac-c562ff80.jpg
                                 FINAL REPORT
 HISTORY:  Multiple sclerosis, presenting with flaring fever.
 
 TECHNIQUE:  Frontal and lateral views of the chest.
 
 COMPARISON:  None.
 
 FINDINGS:
 



In [9]:
batch = next(iter(single_cxrclip_loader))

print(batch["image"].shape)
print(type(batch["text"]))
print(len(batch["text"]))
print(batch["meta"]["image_path"][0])

torch.Size([16, 3, 224, 224])
<class 'list'>
16
/Volumes/TM/Single/images/p10/p10001217/s52067803/a917c883-720a5bbf-02c84fc6-98ad00ac-c562ff80.jpg
